# 🏝 LvTS YouTube Downloader
**by LomaVata Tech Services — *Living With Technology*** · Raiwai, Suva, Fiji

📞 **833 1088** · **746 6941** &nbsp;|&nbsp; ✉️ lomavatatechfiji@gmail.com &nbsp;|&nbsp; 🌐 [fijianblood.github.io/lvts](https://fijianblood.github.io/lvts)
Facebook **@lvtsfiji** &nbsp;|&nbsp; TikTok **@fijianblood8**

---

### How to use (free, nothing to install)
**Coming from the LvTS website?** Your link is already copied, with your quality choice inside it.

1. Sign in with your Google account (top right).
2. Paste the link into **video_url** below.
3. Click **▶ Run**. If Colab shows a warning about this notebook, click **Run anyway**.
4. The first run installs the tools (about 30 seconds). Your browser then downloads the file.

You can also paste any YouTube link here and choose the quality below.
Start from the website: [fijianblood.github.io/lvts/tools/](https://fijianblood.github.io/lvts/tools/#youtube-downloader)

📱 **On a phone?** Choose 1080p, 720p or 480p (MP4 that plays on phones).
**Android:** the video goes to Downloads and shows in Google Photos → Library → Download.
**iPhone:** it goes to the Files app → Downloads; tap it, then Share → Save Video to put it in Photos.

🐍 **This Python script runs here on Google Colab**, Google's cloud computers, signed in with your own Google account. It doesn't run on the LvTS website or on your device.
📲 **The file goes straight to your phone or computer.** Colab only joins the video and sound for a moment, then deletes the file as soon as it reaches you. Nothing is kept on Colab.

⚖️ **Please download only videos you own, have permission to download, or that are licensed for reuse (for example Creative Commons).** Downloading other videos may break YouTube's Terms of Service or copyright law.

*Need a computer fixed, a website, or a video made for your business? Contact LvTS.*

In [ ]:
#@title ▶ LvTS YouTube Downloader – paste your link, then click Run { display-mode: "form" }
video_url = "" #@param {type:"string"}
quality = "720p (recommended)" #@param ["1080p", "720p (recommended)", "480p", "Best available (largest file)", "Audio only (MP3)"]
save_copy_to_google_drive = False #@param {type:"boolean"}
#@markdown A link from the LvTS website already has your quality and Google Drive choice in it.
#@markdown ⚖️ Only download videos you own, have permission for, or that are licensed for reuse.
#@markdown 🐍 This Python script runs on Google Colab with your Google account, not on the LvTS website or your device.

import glob, html, importlib, json, mimetypes, os, re, shutil, subprocess, sys, uuid
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit
from IPython import get_ipython
from IPython.display import HTML, Javascript, display

BRAND_CSS = """<style>
.lvts{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;border-radius:16px;overflow:hidden;max-width:720px;
 box-shadow:0 6px 24px rgba(10,20,60,.25);margin:6px 0}
.lvts-top{background:linear-gradient(135deg,#0b1d4d 0%,#1e40af 60%,#4338ca 100%);color:#fff;padding:18px 22px}
.lvts-top h2{margin:0;font-size:22px;letter-spacing:.3px}
.lvts-top p{margin:4px 0 0;opacity:.9;font-size:14px}
.lvts-tag{display:inline-block;background:#f59e0b;color:#0b1d4d;font-weight:700;border-radius:999px;padding:2px 10px;font-size:12px;margin-bottom:8px}
.lvts-body{background:#fff;color:#0f172a;padding:14px 22px;font-size:14px}
.lvts-foot{background:#0b1d4d;color:#e2e8f0;padding:10px 22px;font-size:13px;line-height:1.6}
.lvts-foot a{color:#fbbf24;text-decoration:none}
.lvts-bar{height:12px;background:#e2e8f0;border-radius:999px;overflow:hidden;margin:8px 0}
.lvts-bar>div{height:100%;background:linear-gradient(90deg,#f59e0b,#fbbf24);transition:width .3s}
.lvts-tip{margin-top:10px;background:#eff6ff;border:1px solid #bfdbfe;border-radius:10px;padding:8px 12px;font-size:13px;color:#1e3a8a;line-height:1.55}
.ok{color:#047857;font-weight:700}.err{color:#b91c1c;font-weight:700}
</style>"""
FOOTER = """<div class="lvts-foot">📞 <b>833 1088</b> · <b>746 6941</b> &nbsp;|&nbsp; ✉️ <a href="mailto:lomavatatechfiji@gmail.com">lomavatatechfiji@gmail.com</a><br>
🌐 <a href="https://fijianblood.github.io/lvts" target="_blank">fijianblood.github.io/lvts</a> &nbsp;|&nbsp; Facebook <b>@lvtsfiji</b> &nbsp;|&nbsp; TikTok <b>@fijianblood8</b></div>"""

def lvts_card(body_html):
    return HTML(BRAND_CSS + '<div class="lvts"><div class="lvts-top"><span class="lvts-tag">JS 🏝 LvTS</span>'
                '<h2>LvTS YouTube Downloader</h2><p>LomaVata Tech Services · Living With Technology · Suva, Fiji</p></div>'
                '<div class="lvts-body">' + body_html + '</div>' + FOOTER + '</div>')

def bar(pct, text):
    return lvts_card(f'{text}<div class="lvts-bar"><div style="width:{pct:.0f}%"></div></div>')

# Quality codes in links from the LvTS website (…?v=…&lvts=720-drive). Keep in step with src/lib/ytLink.ts.
QUALITIES = {"1080": "1080p", "720": "720p (recommended)", "480": "480p",
             "best": "Best available (largest file)", "mp3": "Audio only (MP3)"}
YOUTUBE_LINK = re.compile(r"^https?://(www\.|m\.|music\.)?(youtube\.com|youtu\.be)/")

def read_link(text):
    """Returns (youtube_link, quality, save_to_drive). quality and save_to_drive are None
    unless the link came from the LvTS website. Returns (None, None, None) if it isn't a YouTube link."""
    text = text.strip()
    if not re.match(r"^https?://", text, re.I) and re.match(r"^(www\.|m\.|music\.)?(youtube\.com|youtu\.be)/", text, re.I):
        text = "https://" + text
    if not YOUTUBE_LINK.match(text):
        return None, None, None
    parts = urlsplit(text)
    params = parse_qsl(parts.query, keep_blank_values=True)
    code = next((v for k, v in params if k == "lvts"), None)
    link = urlunsplit(parts._replace(query=urlencode([(k, v) for k, v in params if k != "lvts"])))
    if code is None:
        return link, None, None
    bits = code.lower().split("-")
    return link, QUALITIES.get(bits[0]), "drive" in bits[1:]

def setup_tools(panel):
    """Installs yt-dlp and Deno the first time this runs in a Colab session."""
    if globals().get("_LVTS_READY"):
        return True
    panel.update(bar(0, "⏳ Installing the download tools… (first run only, about 30 seconds)"))
    # yt-dlp[default] includes yt-dlp-ejs; deno is the JavaScript runtime yt-dlp needs for YouTube
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "deno"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        panel.update(lvts_card('<span class="err">❌ Setup failed.</span> Click <b>Runtime → Restart session</b>, then ▶ Run again.'
                               '<pre>' + html.escape(result.stderr[-1500:]) + '</pre>'))
        return False
    import yt_dlp
    importlib.reload(yt_dlp)
    missing = [name for name in ("deno", "ffmpeg") if not shutil.which(name)]
    if missing:
        panel.update(lvts_card('<span class="err">❌ Missing part:</span> ' + " and ".join(missing) +
                               '. Click <b>Runtime → Restart session</b>, then ▶ Run again.'))
        return False
    globals()["_LVTS_READY"] = True
    return True

# Scratch folder: the video and sound are joined here, then the file is sent to the visitor's
# device and deleted. Nothing is kept on Colab.
DOWNLOAD_DIR = "/tmp/lvts_downloads"

SEND_JS = """
(async (id, filename, size, type) => {
  if (!google.colab.kernel.accessAllowed) return;
  const box = document.createElement('div');
  box.style.cssText = 'font:14px system-ui,sans-serif;margin:6px 0;color:#0f172a';
  box.textContent = '📲 Sending "' + filename + '" to this device… ';
  const progress = document.createElement('progress');
  progress.max = size;
  box.appendChild(progress);
  document.body.appendChild(box);
  const parts = [];
  let got = 0;
  const channel = await google.colab.kernel.comms.open(id);
  channel.send({});                       // ready for the first piece
  for await (const message of channel.messages) {
    channel.send({});                     // ready for the next piece
    for (const buffer of message.buffers || []) {
      parts.push(buffer);
      got += buffer.byteLength;
      progress.value = got;
    }
  }
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob(parts, {type}));
  a.download = filename;
  box.appendChild(a);
  a.click();
  box.remove();
})(%s, %s, %d, %s);
"""

def send_to_device(path, on_done):
    """Sends the file to this browser in 1 MB pieces (the same way google.colab.files.download
    does), then deletes it from Colab as soon as the last piece has gone."""
    comm_manager = get_ipython().kernel.comm_manager
    comm_id = "lvts_" + uuid.uuid4().hex
    name, size = os.path.basename(path), os.path.getsize(path)
    kind = mimetypes.guess_type(name)[0] or "application/octet-stream"

    def on_open(comm, _):
        f = open(path, "rb")

        def on_msg(_):
            chunk = f.read(1024 * 1024)
            if chunk:
                comm.send({}, None, [chunk])
                return
            comm.close()
            f.close()
            comm_manager.unregister_target(comm_id, on_open)
            if os.path.exists(path):
                os.remove(path)
            try:
                on_done()
            except Exception:
                pass

        comm.on_msg(on_msg)

    comm_manager.register_target(comm_id, on_open)
    display(Javascript(SEND_JS % (json.dumps(comm_id), json.dumps(name), size, json.dumps(kind))))

class _QuietLogger:  # we show our own friendly messages instead of yt-dlp's raw text
    def debug(self, msg): pass
    def info(self, msg): pass
    def warning(self, msg): pass
    def error(self, msg): pass

def build_options(choice):
    opts = {
        "logger": _QuietLogger(),
        "outtmpl": os.path.join(DOWNLOAD_DIR, "%(title).90B.%(ext)s"),  # file named after the video title
        "noplaylist": True,          # only this video, even if the link is inside a playlist
        "quiet": True, "no_warnings": True, "noprogress": True,
    }
    heights = {"1080p": 1080, "720p (recommended)": 720, "480p": 480}
    if choice in heights:
        h = heights[choice]
        # Prefer H.264 + AAC so the MP4 plays on phones, TVs and Windows
        opts["format"] = (f"bestvideo[height<={h}][vcodec^=avc1]+bestaudio[ext=m4a]/"
                          f"bestvideo[height<={h}]+bestaudio/best[height<={h}]/best")
        opts["merge_output_format"] = "mp4"
    elif choice.startswith("Audio"):
        opts["format"] = "bestaudio/best"
        opts["postprocessors"] = [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}]
    else:
        opts["format"] = "bestvideo+bestaudio/best"
        opts["merge_output_format"] = "mp4"
    return opts

def phone_tip(choice):
    if choice.startswith("Audio"):
        return ('<div class="lvts-tip"><b>📱 On a phone?</b> MP3 files go to Downloads: '
                '<b>Android:</b> Files → Downloads. <b>iPhone:</b> Files app → Downloads.</div>')
    return ('<div class="lvts-tip"><b>📱 On a phone?</b><br>'
            '<b>Android:</b> open <b>Google Photos → Library → Download</b> (or Gallery → Download, or Files → Downloads).<br>'
            '<b>iPhone:</b> open the <b>Files</b> app → <b>Downloads</b>, tap the video, then <b>Share → Save Video</b> to put it in Photos.</div>')

def download(url, choice, to_drive, panel):
    import yt_dlp
    from yt_dlp.utils import DownloadError
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)   # clear anything left from an earlier run
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    state = {"title": ""}
    settings = html.escape(choice) + (" · copy to Google Drive" if to_drive else "")
    panel.update(bar(0, f"⏳ Getting video details…<br><small>{settings}</small>"))

    def on_progress(d):
        state["title"] = d.get("info_dict", {}).get("title", state["title"])
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
            done = d.get("downloaded_bytes", 0)
            pct = done * 100 / total if total else 0
            mb = f"{done/1e6:.1f} MB" + (f" of {total/1e6:.1f} MB" if total else "")
            panel.update(bar(pct, f"⬇️ Downloading <b>{html.escape(state['title'])}</b><br><small>{mb}</small>"))
        elif d["status"] == "finished":
            panel.update(bar(100, "🔧 Finishing the file (joining video and sound)…"))

    opts = build_options(choice)
    opts["progress_hooks"] = [on_progress]
    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
        paths = [d.get("filepath") for d in info.get("requested_downloads", []) if d.get("filepath")]
        path = paths[0] if paths else max(glob.glob(DOWNLOAD_DIR + "/*"), key=os.path.getmtime)
        name = html.escape(os.path.basename(path))
        size = f"{os.path.getsize(path) / 1e6:.1f} MB"
        extra = ""
        if to_drive:
            from google.colab import drive
            drive.mount("/content/drive")
            dest = "/content/drive/MyDrive/LvTS_Downloads"
            os.makedirs(dest, exist_ok=True)
            shutil.copy(path, dest)
            extra = "<br>📁 A copy was also saved to <b>Google Drive → LvTS_Downloads</b>."
        panel.update(lvts_card(f'<span class="ok">✅ Ready:</span> <b>{name}</b> ({size})<br>'
                               '📲 Sending it straight to this phone or computer now. Big files can take a few minutes.'
                               + extra + phone_tip(choice)))

        def sent():
            panel.update(lvts_card(f'<span class="ok">✅ Saved to your device:</span> <b>{name}</b> ({size})<br>'
                                   '🧹 Deleted from Colab. Nothing is kept there.' + extra + phone_tip(choice)))

        send_to_device(path, sent)
    except DownloadError as e:
        shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
        text = str(e)
        if "confirm you" in text and "bot" in text:
            hint = ("YouTube is blocking this Colab computer right now. Try again later, or click "
                    "<b>Runtime → Disconnect and delete runtime</b>, then ▶ Run again to get a different computer.")
        elif "Private video" in text or "members" in text.lower():
            hint = "This video is private or members-only, so it can't be downloaded."
        else:
            hint = "Check the link and try again. If it keeps failing, click <b>Runtime → Restart session</b> and ▶ Run again to update the tools."
        panel.update(lvts_card(f'<span class="err">❌ Download failed.</span> {hint}<br><small>{html.escape(text[:300])}</small>'))
    except Exception as e:  # anything else: show it in the card instead of a long error
        shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
        panel.update(lvts_card(f'<span class="err">❌ Something went wrong.</span> Try ▶ Run again.'
                               f'<br><small>{html.escape(type(e).__name__ + ": " + str(e))[:300]}</small>'))

panel = display(bar(0, "⏳ Getting ready…"), display_id=True)
url, link_quality, link_drive = read_link(video_url)
if not url:
    panel.update(lvts_card('<span class="err">Please paste a YouTube link</span> (it starts with https://www.youtube.com/ or https://youtu.be/) '
                           'into <b>video_url</b>, then click ▶ Run again.<br><small>Get one ready on the LvTS website: '
                           '<a href="https://fijianblood.github.io/lvts/tools/#youtube-downloader" target="_blank">fijianblood.github.io/lvts/tools</a></small>'))
elif setup_tools(panel):
    download(url, link_quality or quality, bool(save_copy_to_google_drive or link_drive), panel)